# Semantic correspondence: Colab runner

This notebook sets options and calls the repo code (`scripts/colab_setup.py`, `run.py`).
Code changes belong in the repo: edits made here are lost. Full guide: `RUN.md`.

**Before the first run:**
1. Runtime > Change runtime type > **GPU** (needed for `train`; `evaluate`/`submit` also work on CPU, slowly).
2. Accept the competition rules on Kaggle.
3. Add to Colab Secrets (key icon) with Notebook access ON: `GITHUB_TOKEN`, `KAGGLE_KEY` (and `KAGGLE_USERNAME` only for legacy keys; `HF_TOKEN` / `WANDB_API_KEY` are optional).

In [ ]:
# Run settings
BRANCH = "giuseppe"      # branch to run; push your changes there first
MODE = "train"           # "train" | "evaluate" | "submit"   (settings: BASE in src/config.py)
RUN = ""                 # evaluate/submit an existing run (uses its best.pt)
RESUME = ""              # train: continue this run id after a disconnect
EXTRA = ""               # other run.py flags, e.g. "--dry-run"

# Unified preset & transformation switches (passed via CLI flags so src/config.py stays clean for git pull):
PRESET = "unified_lora"     # "unified_lora" (LoRA + 92x92 loss + adaptive radius + zoom) | "none" (BASE)
OBJECT_ZOOM = True          # 2-stage object-centric zoom at inference (src/models/zoom.py)
CROSS_TRANSFORM = False     # src/models/cross_transform.py (off by default: overfits 314 pairs)
CONSTELLATION_ALIGN = False # src/models/constellation.py (pair-level Sim(2) prior at inference)
EVAL_IMAGE_SIZE = None      # e.g. 840 for higher-res evaluate/submit (None = use run's 644)
FUSION = "none"             # "none" (fast pure DINOv2) | "concat" | "adapter"

In [ ]:
# Clone the private repo (using GITHUB_TOKEN secret) and run scripts/colab_setup.py:
# pull, install environment.yml (pip part), secrets, Drive, data and weights.
import os
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN").strip()
    REPO_URL = f"https://{token}@github.com/julpfi/sapienza_aml_semantic_correspondence.git"
except Exception:
    REPO_URL = "https://github.com/julpfi/sapienza_aml_semantic_correspondence.git"

# Support new Kaggle API tokens saved in KAGGLE_KEY without KAGGLE_USERNAME
try:
    k_key = userdata.get("KAGGLE_KEY").strip()
    try:
        userdata.get("KAGGLE_USERNAME")
    except Exception:
        if k_key and not os.environ.get("KAGGLE_API_TOKEN"):
            os.environ["KAGGLE_API_TOKEN"] = k_key
except Exception:
    pass

REPO_DIR = "/content/sapienza_aml_semantic_correspondence"
if not os.path.isdir(f"{REPO_DIR}/.git"):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git checkout -- src/config.py 2>/dev/null || true
%run scripts/colab_setup.py --branch {BRANCH}

In [ ]:
# Start the run. Runs are saved on your Drive: aml_semantic_correspondence/outputs/<run_id>
flags = f"--mode {MODE} {EXTRA}"
if RESUME:
    flags += f" --resume {RESUME}"
elif RUN:
    flags += f" --run {RUN}"
elif PRESET and PRESET != "none":
    flags += f" --preset {PRESET}"
if MODE == "train":
    flags += " --require-gpu"
    if not RESUME:
        flags += " --cross-transform" if CROSS_TRANSFORM else " --no-cross-transform"
        flags += f" --fusion {FUSION}"
else:
    if EVAL_IMAGE_SIZE:
        flags += f" --image-size {EVAL_IMAGE_SIZE}"
flags += " --object-zoom" if OBJECT_ZOOM else " --no-object-zoom"
flags += " --constellation-align" if CONSTELLATION_ALIGN else " --no-constellation-align"
print("Running: python run.py", flags)
!python run.py {flags}

In [ ]:
# Optional: paired comparison of evaluated runs (seeds comma-separated, A then B), e.g. "a_s0,a_s1 b_s0,b_s1".
# Each run needs per_query.csv: MODE = "evaluate" with RUN = "<run_id>" writes it.
COMPARE = ""
if COMPARE:
    !python scripts/compare_runs.py {COMPARE}

In [ ]:
# Optional: upload a submission to Kaggle (set the run id printed above).
SUBMIT_RUN = ""  # set to the run id you want to submit, or "" to skip
if SUBMIT_RUN:
    !python scripts/kaggle_io.py submit --file /content/drive/MyDrive/aml_semantic_correspondence/outputs/{SUBMIT_RUN}/submission.csv --message "{SUBMIT_RUN}"